In [ ]:
import pandas as pd

df = pd.read_parquet("data/df_train_full_sampled.parquet")
df.shape

In [ ]:
display(df.head(5))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, )

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

drop_cols = ["case_id", "WEEK_NUM", "target"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

cat_cols = X_train.select_dtypes(
    include=["object", "category"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

logreg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

logreg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

y_pred_proba = logreg.predict_proba(X_test)[:, 1]
y_pred = logreg.predict(X_test)

print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

In [ ]:
import numpy as np


def gini_stability(week_num, target, score, w_fallingrate=88.0, w_resstd=-0.5):
    base = pd.DataFrame(
        {"WEEK_NUM": week_num, "target": target, "score": score})

    # weeks with only one target class have undefined AUC, so they're dropped
    valid_weeks = base.groupby("WEEK_NUM")["target"].nunique()
    valid_weeks = valid_weeks[valid_weeks > 1].index
    dropped = base["WEEK_NUM"].nunique() - len(valid_weeks)
    if dropped:
        print(
            f"Dropping {dropped} week(s) with a single target class (undefined AUC)")
    base = base[base["WEEK_NUM"].isin(valid_weeks)]

    weeks = np.sort(base["WEEK_NUM"].unique())
    gini_in_time = (
        base.groupby("WEEK_NUM")[["target", "score"]]
        .apply(lambda x: 2 * roc_auc_score(x["target"], x["score"]) - 1)
        .sort_index()
        .to_numpy()
    )

    x = np.arange(len(gini_in_time))
    a, b = np.polyfit(x, gini_in_time, 1)
    residuals = gini_in_time - (a * x + b)

    avg_gini = np.mean(gini_in_time)
    falling_rate = min(0, a)
    res_std = np.std(residuals)

    stability_score = avg_gini + w_fallingrate * falling_rate + w_resstd * res_std
    return stability_score, avg_gini, a, b, res_std, weeks, gini_in_time


stability_score, avg_gini, slope_a, intercept_b, res_std, weeks, gini_in_time = gini_stability(
    test_df["WEEK_NUM"], y_test, y_pred_proba
)

print("Mean Gini:", avg_gini)
print("Slope (a):", slope_a)
print("Residual std:", res_std)
print("Stability metric:", stability_score)

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(weeks))
trend_line = slope_a * x + intercept_b

plt.figure(figsize=(10, 5))
plt.plot(weeks, gini_in_time, marker="o", label="Gini per week")
plt.plot(weeks, trend_line, linestyle="--", color="red",
         label=f"Trend (slope={slope_a:.4f})")
plt.xlabel("WEEK_NUM")
plt.ylabel("Gini")
plt.title("Weekly Gini with fitted trend line")
plt.legend()
plt.grid(alpha=0.3)
plt.show()